# 02 · Models & materializations

A dbt **model** is one `SELECT` statement in a `.sql` file. The
**materialization** decides what dbt turns it into:

| materialization | becomes | rebuilt | in this project |
|---|---|---|---|
| `view` | `CREATE VIEW` | every query | all of staging |
| `table` | `CREATE TABLE AS` | every `dbt run` | `dim_customers`, ... |
| `ephemeral` | nothing -- inlined as a CTE | n/a | `int_order_items_enriched` |
| `incremental` | table, updated in place | only new/changed rows | `fct_orders`, `fct_events` |
| `materialized_view` | PG materialized view | `REFRESH` on each run | `agg_daily_revenue` |

Companion reading: [docs/04](../docs/04_models_and_materializations.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Build one model of each kind

In [ ]:
res = run_dbt(["run", "--select", "stg_customers", "dim_customers", "agg_daily_revenue", "fct_orders"])
assert res.success

## 2. Ask Postgres what each one actually IS

`pg_class.relkind`: `v` = view, `r` = ordinary table, `m` = materialized view.

In [ ]:
%%sql
select n.nspname as schema, c.relname as relation, c.relkind
from pg_class c
join pg_namespace n on n.oid = c.relnamespace
where n.nspname like 'dev%'
  and c.relname in ('stg_customers', 'dim_customers', 'agg_daily_revenue',
                    'fct_orders', 'int_order_items_enriched')
order by c.relname

Notice what is **missing**: `int_order_items_enriched` (the ephemeral model)
does not exist in the warehouse at all.

## 3. Compiled vs run SQL

For every model dbt writes two files under `target/`:

* `target/compiled/.../<model>.sql` -- your SELECT after Jinja rendering
* `target/run/.../<model>.sql` -- that SELECT wrapped in the materialization DDL

In [ ]:
compiled = Path("target/compiled/jaffle_shop/models/staging/stg_customers.sql")
run_sql  = Path("target/run/jaffle_shop/models/staging/stg_customers.sql")

print("--- COMPILED (your logic) ---")
print(compiled.read_text()[:400])
print("\n--- RUN (what hit the warehouse) ---")
print(run_sql.read_text()[:400])

## 4. See the ephemeral model get inlined

In [ ]:
fct = Path("target/compiled/jaffle_shop/models/marts/fct_orders.sql").read_text()
start = fct.find("__dbt__cte__int_order_items_enriched")
print(fct[max(0, start - 60):start + 500])

dbt spliced the ephemeral model in as `__dbt__cte__int_order_items_enriched`.
Every model that refs it gets its own copy -- shared logic, recomputed per
consumer. That is the trade-off.

## 5. Materialized view: refresh, don't rebuild

Run it again and read the message: dbt issues `REFRESH MATERIALIZED VIEW`,
not `CREATE`. The `on_configuration_change` config governs what happens if
you edit the model's *definition*.

In [ ]:
res = run_dbt(["run", "--select", "agg_daily_revenue"])
assert res.success
print(res.result.results[0].message)

## 6. Where can materialization be configured?

Three places, most specific wins:

1. `dbt_project.yml` -- folder-level defaults (staging = view, marts = table)
2. properties `.yml` -- per model under `config:`
3. the model file itself -- `config(materialized='...')` at the top

Try it: change `dim_products` to a view with a config block, `dbt run
--select dim_products`, check `relkind` above, then revert.